# 9.0 - Domain Adaptation (synthetic shift)

The ADAPTATION axis. UCF101 is clean and centred; real-world video (compression, lighting,
low resolution) is not. Live YouTube downloads are unreliable on Colab (dead URLs + bot
checks), so the domain shift is SIMULATED with a controlled corruption - brightness/contrast
change, resolution loss and blur - applied to the base classes. This is the ImageNet-C style
of domain-shift study: reproducible, and it isolates exactly what a FROZEN backbone can and
cannot adapt to.

State the limit up front: the ResNet is frozen, so it cannot undo low-level corruption. The
LSTM+head can absorb a roughly linear feature shift; they cannot recover what the conv layers
discarded. This measures how much of the gap the trainable part closes, while replaying clean
exemplars so the clean domain is not forgotten.

## **Connect Colab**

In [ ]:
import os, sys, subprocess
from getpass import getpass

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

REPO   = "silvergjeka22/Unified-Lifelong-Action-Learning"
BRANCH = "embeddings"
ROOT   = "/content/ulal"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token: ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [ ]:
os.environ["KAGGLE_USERNAME"] = "colab"
os.environ["KAGGLE_KEY"]      = "PASTE_YOUR_KAGGLE_KEY"

## **Download DATASET**

The corruption is applied to raw frames, so the dataset is downloaded here.

In [ ]:
from src.bootstrap import setup
setup(drive=True, dataset=True)

In [ ]:
# imports
%run /content/ulal/src/imports.py
set_seed(cfg.SEED)

## **Load Cache, Head, Backbone**

The base head is the model being adapted; the frozen ResNet backbone re-extracts features
from both clean and corrupted clips.

In [ ]:
CACHE, LSTM_STATE, META = load_feature_cache(cfg.FEAT_CACHE)
class_to_idx = {cls_name: i for i, cls_name in enumerate(cfg.SELECTED_CLASSES)}

head = load_or_train_base_head(f"{cfg.CKPT_DIR}/head_base.pt", CACHE, LSTM_STATE, len(cfg.SELECTED_CLASSES), device)

cfg.RESNET50_PATH = f"{cfg.DRIVE_PROJECT}/resnet50/models/ResNet50_10C_groupsplit.pth"
backbone = load_frozen_backbone(device, ResNet50LSTMTeacher, cfg.RESNET50_PATH,
                                hidden_size=cfg.TEACHER_HIDDEN, num_classes=len(cfg.SELECTED_CLASSES),
                                dropout_p=cfg.DROPOUT_P, remap_fn=remap_teacher_checkpoint)
print("head + frozen backbone ready")

## **Build the Shifted Domain**

Rebuild the base clips, then extract features twice: once clean (from the cache) and once
through the corruption. The corrupted features are the out-of-domain data.

In [ ]:
base_split = build_group_split(cfg.SELECTED_CLASSES, train_groups=18, val_groups=3)
preprocess_group_split(base_split, cfg.SELECTED_CLASSES, output_root=cfg.BASE_ROOT)

PIN = device.type == "cuda"
base_train_ds, base_train_loader = build_clip_loader(cfg.BASE_ROOT, cfg.SELECTED_CLASSES, class_to_idx, "train", cfg.BATCH_SIZE, cfg.NUM_WORKERS, PIN)
base_test_ds,  base_test_loader  = build_clip_loader(cfg.BASE_ROOT, cfg.SELECTED_CLASSES, class_to_idx, "test",  cfg.BATCH_SIZE, cfg.NUM_WORKERS, PIN)

In [ ]:
BLUR, BRIGHT, CONTRAST, DOWNSCALE = 1.2, 0.75, 0.8, 0.5

shift_train_s, shift_train_y = extract_shifted_features(backbone, base_train_loader, device, cfg.BACKBONE_DIM, torch.float32, BLUR, BRIGHT, CONTRAST, DOWNSCALE)
shift_test_s,  shift_test_y  = extract_shifted_features(backbone, base_test_loader,  device, cfg.BACKBONE_DIM, torch.float32, BLUR, BRIGHT, CONTRAST, DOWNSCALE)
print(f"shifted train {tuple(shift_train_s.shape)}  test {tuple(shift_test_s.shape)}")

See the shift: the same clip, clean (top) vs corrupted (bottom).

In [ ]:
clip = torch.load(base_test_ds.samples[0][0])
corrupt = domain_corrupt(clip.clone(), BLUR, BRIGHT, CONTRAST, DOWNSCALE)
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

fig, axes = plt.subplots(2, 4, figsize=(11, 5.5))
for col, f in enumerate([0, 5, 10, 15]):
    for row, (img, name) in enumerate([(clip, "clean"), (corrupt, "shifted")]):
        fr = (img[f] * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
        axes[row, col].imshow(fr)
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_title(name, loc="left")

plt.tight_layout()
plt.savefig(f"{cfg.RESULTS_DIR}/stage9_domain_shift.png", dpi=150, bbox_inches="tight")
plt.show()

## **Domain Gap (before adaptation)**

The UCF-trained head on clean vs shifted test clips - the gap is the domain shift.

In [ ]:
clean_test_s, clean_test_y = CACHE["t0_test"]
clean_before = eval_head(head, clean_test_s, clean_test_y, device)
shift_before = eval_head(head, shift_test_s, shift_test_y, device)
print(f"before | clean UCF: {clean_before:.2%} | shifted: {shift_before:.2%} | gap: {clean_before-shift_before:.2%}")

In [ ]:
clean_emb, clean_ey = head_embeddings(head, CACHE, [0], device, "test")
shift_emb = head.embed(shift_test_s.to(device)).cpu()
clean_stats = compute_class_stats(clean_emb, clean_ey)
shift_stats = compute_class_stats(shift_emb, shift_test_y)
shifts = measure_shift(clean_stats, shift_stats, cfg.SELECTED_CLASSES)

## **Active Learning**

Score the shifted training clips by predictive entropy and keep the most informative
`AL_BUDGET_PER_CLASS` per class - the clips worth labelling.

In [ ]:
shift_by_class = {}
for cls_name in cfg.SELECTED_CLASSES:
    idx = class_to_idx[cls_name]
    shift_by_class[cls_name] = shift_train_s[shift_train_y == idx]

selected = select_top_k(shift_by_class, cfg.AL_STRATEGY, cfg.AL_BUDGET_PER_CLASS, head=head, device=device)

In [ ]:
al_list, al_ylist = [], []
for cls_name, idxs in selected.items():
    al_list.append(shift_by_class[cls_name][idxs])
    al_ylist.extend([class_to_idx[cls_name]] * len(idxs))

al_s = torch.cat(al_list)
al_y = torch.tensor(al_ylist, dtype=torch.long)
al_t = head.embed(al_s.to(device)).cpu()
print(f"AL-selected shifted clips: {tuple(al_s.shape)}")

## **Reptile Adaptation**

Fill a replay buffer with clean UCF exemplars, then Reptile-adapt a copy of the head on the
AL clips mixed with replay. The replay is what keeps the clean domain from being forgotten.

In [ ]:
buffer = SmartReplayBuffer(max_per_class=15)
base_s, base_yb = CACHE["t0_train"]
base_emb_all, _ = head_embeddings(head, CACHE, [0], device, "train")

for cls_name in cfg.SELECTED_CLASSES:
    idx  = class_to_idx[cls_name]
    mask = base_yb == idx
    buffer.add_class(idx, base_s[mask], base_emb_all[mask], strategy="hard")

print(f"buffer: {len(buffer.s_store)} classes")

In [ ]:
import copy
set_seed(cfg.SEED)
adapted = copy.deepcopy(head)

reptile_adapt(adapted, al_s, al_t, al_y, buffer, device,
              episodes=cfg.ADAPT_EPISODES, inner_lr=cfg.ADAPT_INNER_LR,
              inner_steps=cfg.ADAPT_INNER_STEPS, epsilon=cfg.ADAPT_EPSILON,
              mse_weight=0.5, eval_s=shift_test_s, eval_y=shift_test_y)

## **After adaptation**

In [ ]:
clean_after = eval_head(adapted, clean_test_s, clean_test_y, device)
shift_after = eval_head(adapted, shift_test_s, shift_test_y, device)
print(f"shifted : {shift_before:.2%} -> {shift_after:.2%}  (gain {shift_after-shift_before:+.2%})")
print(f"clean   : {clean_before:.2%} -> {clean_after:.2%}  (change {clean_after-clean_before:+.2%})")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
groups = ["shifted", "clean UCF"]
before = [shift_before * 100, clean_before * 100]
after  = [shift_after * 100,  clean_after * 100]
x = np.arange(len(groups))

ax.bar(x - 0.2, before, 0.4, label="before")
ax.bar(x + 0.2, after,  0.4, label="after")
ax.set_xticks(x); ax.set_xticklabels(groups)
ax.set_ylabel("accuracy (%)")
ax.set_title("Domain adaptation: gain vs forgetting")
ax.legend()

plt.tight_layout()
plt.savefig(f"{cfg.RESULTS_DIR}/stage9_adaptation.png", dpi=150, bbox_inches="tight")
plt.show()

## **Save**

In [ ]:
payload = {
    "stage":        "9_domain_adapt_synthetic",
    "corruption":   {"blur": BLUR, "brightness": BRIGHT, "contrast": CONTRAST, "downscale": DOWNSCALE},
    "clean_before": float(clean_before), "clean_after": float(clean_after),
    "shift_before": float(shift_before), "shift_after": float(shift_after),
}
os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
with open(f"{cfg.RESULTS_DIR}/stage9_da.json", "w") as f:
    json.dump(payload, f, indent=2, default=float)

torch.save(adapted.state_dict(), f"{cfg.CKPT_DIR}/head_adapted_shift.pt")
print(f"Saved -> {cfg.RESULTS_DIR}/stage9_da.json")